# 🧹 Notebook 02: Text Preprocessing & Vectorization

In this notebook, we detail the text preprocessing steps required to clean raw messages and the **TF-IDF Vectorization** process that converts text into numeric feature vectors.

In [1]:
import os
import sys
import pandas as pd
import re
import nltk

# Add src to import clean_text
sys.path.append(os.path.abspath('../src'))
from preprocess import clean_text, stemmer, stop_words

## 1. Text Preprocessing Workflow

Text preprocessing is critical for natural language processing. It involves:
1. **Lowercasing**: Converting all letters to lower case to treat words like "Free" and "free" identically.
2. **Punctuation & Digit Removal**: Removing noise such as `!`, `?`, or numbers that can complicate tokenization.
3. **Stopword Removal**: Excluding common english words like "the", "is", "in" which don't carry distinct meaning for classification.
4. **Stemming**: Reducing words to their root form (e.g. "winner", "winning", "won" all reduce to "win").

In [2]:
sample_msg = "WINNER! You have won a free holiday, text CLAIM to 88310 now!"
print("Raw Message:", sample_msg)

# Step 1: Lowercase
step1 = sample_msg.lower()
print("\n1. Lowercased:", step1)

# Step 2: Remove non-alphabetic chars
step2 = re.sub(r'[^a-z\s]', ' ', step1)
print("2. Punctuation & digits removed:", step2)

# Step 3: Stopword removal
words = step2.split()
step3 = [w for w in words if w not in stop_words]
print("3. Stopwords removed:", step3)

# Step 4: Stemming
step4 = [stemmer.stem(w) for w in step3]
print("4. Stemmed:", step4)

# Rejoined
final_cleaned = " ".join(step4)
print("\nFinal preprocessed representation:")
print(final_cleaned)

Raw Message: WINNER! You have won a free holiday, text CLAIM to 88310 now!

1. Lowercased: winner! you have won a free holiday, text claim to 88310 now!
2. Punctuation & digits removed: winner  you have won a free holiday  text claim to       now 
3. Stopwords removed: ['winner', 'free', 'holiday', 'text', 'claim']
4. Stemmed: ['winner', 'free', 'holiday', 'text', 'claim']

Final preprocessed representation:
winner free holiday text claim


## 2. Reusable clean_text Module

We created a reusable module `src/preprocess.py` for this pipeline. Let's verify it on some test data.

In [3]:
test_messages = [
    "Urgent! call 09066362231 to claim your prize!",
    "Hey, how are you? Let's meet tomorrow.",
    "Free entry in 2 a wkly comp to win FA Cup final..."
]

for raw in test_messages:
    print(f"Raw: {raw}")
    print(f"Cleaned: {clean_text(raw)}")
    print("-" * 50)

Raw: Urgent! call 09066362231 to claim your prize!
Cleaned: urgent call claim prize
--------------------------------------------------
Raw: Hey, how are you? Let's meet tomorrow.
Cleaned: hey let meet tomorrow
--------------------------------------------------
Raw: Free entry in 2 a wkly comp to win FA Cup final...
Cleaned: free entri wkli comp win fa cup final
--------------------------------------------------


## 3. TF-IDF Vectorization

**TF-IDF** (Term Frequency - Inverse Document Frequency) converts text into numbers by weighting word frequency in a document against its frequency in the entire corpus.
- **Term Frequency (TF)**: How often a word appears in a specific message.
- **Inverse Document Frequency (IDF)**: Penalizes words that appear very frequently across all documents (e.g. "go", "get").

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Small example corpus
corpus = [
    "free claim prize",
    "meet tomorrow lunch",
    "claim free lunch free"
]

vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(corpus)

# Convert to dense DataFrame for presentation
tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray(), 
    columns=vectorizer.get_feature_names_out(),
    index=[f"Doc {i+1}" for i in range(len(corpus))]
)

print("Vocabulary mapping:")
print(vectorizer.vocabulary_)
print("\nTF-IDF Matrix:")
display(tfidf_df)

Vocabulary mapping:
{'free': 1, 'claim': 0, 'prize': 4, 'meet': 3, 'tomorrow': 5, 'lunch': 2}

TF-IDF Matrix:


,claim,free,lunch,meet,prize,tomorrow
Doc 1,0.517856,0.517856,0.000000,0.000000,0.680919,0.000000
Doc 2,0.000000,0.000000,0.473630,0.622766,0.000000,0.622766
Doc 3,0.408248,0.816497,0.408248,0.000000,0.000000,0.000000
